# MetaCog-Bench · Getting Started（入门 / 基线笔记）

本笔记演示如何**本地或 Kaggle 上**跑通整个基准流水线：加载题集 → 生成提交 → 评分出榜。
默认用仓库自带的 `sample_submission.csv`（DeepSeek V4.1 Flash 真实预测）直接跑通，**零 API / 零 GPU**。
把 `DEMO` 改成 `False` 并在 `predict()` 里接入你自己的模型即可提交你的成绩。

> 详细文档见同目录 `README.md`；完整提案见仓库根 `FengShuo_C2A_proposal.md`。

## 0. 环境 / 路径

- **本地**：直接运行（仓库已含 `src/scoring.py` 与 `dataset/`）。
- **Kaggle**：首次运行会 `git clone` 仓库以拿到评分代码；记得在 Notebook 设置里**开启 Internet**。

In [ ]:
import os, sys, csv, shutil, subprocess

ON_KAGGLE = os.path.exists("/kaggle/input")

if ON_KAGGLE:
    # 克隆仓库拿到评分代码（scoring.py / benchmark_utils.py / src/scoring.py）
    if not os.path.exists("AI-plus-X-C2A-C9"):
        !git clone https://github.com/Feng11-kyrie/AI-plus-X-C2A-C9
    REPO = "AI-plus-X-C2A-C9"
    # 关联你的 Dataset 后，把下面路径改成 /kaggle/input/<你的dataset名>
    DS = os.path.join(REPO, "kaggle", "benchmark", "dataset")
    BENCH = os.path.join(REPO, "kaggle", "benchmark")
else:
    # 本地：从当前工作目录向上查找包含 kaggle/benchmark/dataset 的仓库根
    _cand = os.getcwd()
    ROOT = _cand
    for _ in range(6):
        if os.path.exists(os.path.join(_cand, "kaggle", "benchmark", "dataset")):
            ROOT = _cand
            break
        _cand = os.path.dirname(_cand)
    BENCH = os.path.join(ROOT, "kaggle", "benchmark")
    DS = os.path.join(BENCH, "dataset")

sys.path.insert(0, BENCH)
print("ON_KAGGLE =", ON_KAGGLE)
print("dataset dir :", DS)
print("benchmark dir:", BENCH)

## 1. 加载题集与金标准

- `metacog_bench.csv`：**公开**题集（id, task, domain, prompt），参赛者可见。
- `ground_truth.csv`：**隐藏**金标准（评分端用，含 gold_answer / is_ood / is_answerable）。
- `public_private_split.csv`：公开/私有榜切分。

In [ ]:
import pandas as pd

bench = pd.read_csv(os.path.join(DS, "metacog_bench.csv"))
gt    = pd.read_csv(os.path.join(DS, "ground_truth.csv"))
split = pd.read_csv(os.path.join(DS, "public_private_split.csv"))

print("公开题集:", bench.shape, "| 金标准:", gt.shape, "| 切分:", split.shape)
bench.head()

## 2. 标准 Prompt 与你的 `predict()`

基准**固定**下面这套标准 prompt（提案 §2.1 承诺、prompt 敏感性报告实测 4 套 prompt 下 OOD 护栏稳健）。
请让你的模型严格按该格式输出，便于解析器（行首锚定）稳定抽取字段。

In [ ]:
from benchmark_utils import STANDARD_PROMPT
print(STANDARD_PROMPT)

def predict(item):
    """你的模型推理函数。

    输入：一道题 dict（含 'prompt' 等）
    输出：dict {answer_text, confidence(0-100), self_pred(0-100), can_answer('YES'/'NO')}
    默认占位实现：一律拒答——仅用于跑通格式。请替换为真实模型调用。
    """
    # TODO: 构造 prompt = STANDARD_PROMPT.format(q=item['prompt'])，调用你的大模型，
    #       解析输出得到 answer_text / confidence / self_pred / can_answer。
    return {"answer_text": "I CANNOT ANSWER", "confidence": 50,
            "self_pred": 50, "can_answer": "NO"}

## 3. 生成提交文件（submission.csv）

`DEMO=True`：直接用仓库自带的真实预测样例跑通；`DEMO=False`：用上面的 `predict()` 对你的模型逐题推理生成。

In [ ]:
SUB_PATH = os.path.join(BENCH, "submission.csv")
DEMO = True   # 改 False 接入你自己的模型

if DEMO:
    shutil.copy(os.path.join(DS, "sample_submission.csv"), SUB_PATH)
    print("已使用示例提交（DeepSeek V4.1 Flash 真实预测）:", SUB_PATH)
else:
    rows = []
    for _, it in bench.iterrows():
        p = predict(it.to_dict())
        rows.append([it["id"], p["answer_text"], p["confidence"],
                     p["self_pred"], p["can_answer"]])
    with open(SUB_PATH, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["id", "answer_text", "confidence", "self_pred", "can_answer"])
        w.writerows(rows)
    print("已用 predict() 生成提交:", SUB_PATH, "共", len(rows), "行")

print("--- 前 3 行 ---")
print(open(SUB_PATH, encoding="utf-8").read().splitlines()[:4])

## 4. 评分 & 排行榜

评分端**自行**用 `ground_truth.csv` 判分（不信任提交里的正确性），再调用已验证的 `src/scoring.py`。
输出各任务原始指标 + 公开/私有 `metacog_score`（合成分，仅用于排序）。

In [ ]:
from scoring import score

rep = score(SUB_PATH,
            os.path.join(DS, "ground_truth.csv"),
            os.path.join(DS, "public_private_split.csv"),
            os.path.join(BENCH, "scores.csv"))

print("\n=== 排行榜条目 (scores.csv) ===")
display(pd.read_csv(os.path.join(BENCH, "scores.csv")))

## 5. 提交到 Kaggle（正式竞赛模式）

1. 新建 **Dataset**，上传 `dataset/` 下 4 个 CSV（`metacog_bench.csv` 公开；其余评分端用）。
2. 复制本笔记为 **Notebook**，`Add Data` 关联上一步 Dataset，点 Run 即可看到全流水线。
3. 若做正式竞赛：把 `scoring.py` + `ground_truth.csv` + `public_private_split.csv` 放进平台**隐藏评分容器**，只对参赛者暴露 `metacog_bench.csv` 与 `sample_submission.csv`。

完整说明见 `README.md`。指标定义与权重：

- `metacog_score = 100 × (0.30·A + 0.35·B + 0.20·C + 0.15·G)`
- `A` = (meta_auroc−0.5)/0.5（截 [0,1]）　`B` = 1−整体ECE　`C` = (Omniscience+0.5)/1.5　`G` = calibration_gap/0.5
- 全部任务**原始指标一并公开**，避免「单一数字误导」。